# Building AI-Friendly Scientific Software: A Model Context Protocol Journey

In this post I'll share my recent (soon to be outdated) experience extending and maintaining a scientific codebase, [Napistu](https://github.com/napistu/napistu) using modern agentic code development (Cursor & Claude). 

## The AI development paradox

🚀 These approaches are amazing! They can massively increase productivity by efficiently tackling routine tasks, lowering the activation energy for skill development and simplifying debugging. These contributions not only save devs time, they also preserve our mental energy for critical tasks - I love just throwing a set of pytest failures at an agent and saying "you deal with it".

⚠️ But, this hands-off, vibe coding, mentality can easily get you into trouble. Its easy for AI agents to not follow common design patterns, to duplicate code, to over-engineer irrelevant features, and to produce straight-up brittle, buggy code. This can easily render a project unmaintainable.

For example, here's what happened when I asked Cursor to **add validation to the SBML parsing function** without any context about our existing patterns:

```python
def validate_sbml_file(file_path: str) -> bool:
    """Generic SBML validation using libsbml."""
    try:
        import libsbml
        document = libsbml.readSBML(file_path)
        if document.getNumErrors() > 0:
            return False
        return True
    except Exception:
        return False
```

The AI-generated code wasn't wrong, but it completely missed our existing SBML_dfs class and our domain-specific validation requirements. Multiply this across dozens of functions and you end up with a fragmented codebase.

🎯 Many of us are trying to navigate this tension - looking for a sweet spot where AI maximizes productivity. We are also looking for how to shift this sweet spot - finding tasks where agent performance can be shifted from dubious to quality.

**How do we get AI to understand our domain-specific codebase?**

Here, I'll share some approaches that I've found particularly helpful for scientific programming in the age of AI. A critical challenge is providing the right information up-front and allowing agents to surface relevant information on-demand. To do this we can use the Model Context Protocol (MCP) to provide information in a format suitable for AI models. In this post, I'll discuss the development of a Napistu MCP server, how I deployed it to Cloud Run via Github Actions, and provide some case studies of how agents operate with and without using the MCP server.

## Information is everything

**The core problem**: The core issue affects any domain-specific codebase - whether it's a financial trading system, game engine, or scientific library. AI agents lack the contextual knowledge to understand existing patterns, conventions, and domain-specific approaches.

Let's trace through how this plays out with a real example. Say I want to understand: "How do I create a consensus network from multiple pathway databases in Napistu?"

**Without any context**, Claude gives me generic advice:

    "To create a consensus network, you'll typically want to merge graph structures. You can use libraries like NetworkX to combine multiple graphs, then apply consensus algorithms..."

**With relevant code context** (say, I paste some Napistu files into Claude's context), the response improves:

    "Looking at your codebase, I see you have an SBML_dfs class. You could extend it to merge multiple data sources..."

But even modern tools with large context windows hit limits here - you can't fit an entire mature codebase, and you have to guess what's relevant upfront.

**An expert's response** draws from multiple information sources:

    "Check out the 'merging_models_into_a_consensus' tutorial - it walks through exactly this workflow. The key is using consensus.create_network() with multiple SBML_dfs objects. There's also a recent GitHub issue (#73) where someone extended this for metabolic networks, and the wiki has details on handling database-specific ID mapping."

The difference is stark: the expert seamlessly integrates code patterns, tutorials, community discussions, and implementation examples. The challenge isn't just context window limits - it's **information fragmentation** (knowledge scattered across repositories, wikis, issues, tutorials) and **signal vs. noise** (finding the exact relevant pattern among thousands of functions).

### Solution preview: What if an AI could retrieve domain-specific information on demand?

First, I can install Napistu with MCP dependencies enabled

```bash
pip install 'napistu[mcp]'
```

In [ ]:
from napistu.mcp import client
config = production_client_config()
result = await search_component("codebase", "SBML_dfs validation methods", config=config)
# Returns actual Napistu function signatures, docs, and usage examples

ModuleNotFoundError: No module named 'fastmcp'


This simple interaction transforms how AI agents understand and work with Napistu. Instead of hallucinating generic solutions, they can discover actual patterns, find relevant tutorials, and understand our domain-specific approaches.

### Enter MCP

Model Context Protocol (MCP) provides a standardized way for AI models to access external information sources. Think of MCP as giving AI agents a research assistant who knows your project inside and out - someone who can instantly find relevant documentation, code examples, and implementation patterns specific to your domain.

# Anatomy of the Napistu MCP Server

## FastMCP Foundation

The Model Context Protocol provides a standard way for AI models to access external information. [FastMCP](https://github.com/jlowin/fastmcp) gives us a Flask-like Python implementation:

```python
from fastmcp import FastMCP

mcp = FastMCP("napistu-server")

@mcp.resource("napistu://health")
async def health_check():
    return {"status": "healthy", "components": [...]}

@mcp.tool()
async def search_documentation(query: str) -> dict:
    return {"results": [...]}
```

FastMCP handles the protocol details - we focus on exposing Napistu's knowledge.

## Components

Napistu uses a component-based architecture which provides separation of concerns (each component manages its own data), graceful degradation (failed components don't break others), and flexible deployment (enable only needed components). This also lets me create servers which are tailored to different use cases (e.g., a local server which can execute Napistu code or a remote docs server).

The current components are:
- Documentation: READMEs, wiki pages, GitHub issues/PRs via API  
- Codebase: API docs and function signatures from ReadTheDocs  
- Tutorials: Jupyter notebooks converted to searchable markdown  
- Execution: Working with a live Python environment (in development)
- Health: Server monitoring and component diagnostics

Each component follows a consistent pattern: load data, register endpoints, handle search:

```python
class DocumentationComponent(MCPComponent):
    async def initialize(self, semantic_search: SemanticSearch = None) -> bool:
        """Load READMEs, wiki pages, GitHub issues"""
        # Load external data and populate component state
        return success
    
    def register(self, mcp: FastMCP) -> None:
        """Register resources and tools with MCP server"""
        @mcp.tool()
        async def search_documentation(query: str):
            return self.state.semantic_search.search(query, "documentation")
```

## Smart Search: Semantic + Vector Embeddings

We can search content using exact keywords (e.g., "create_consensus") or semantic search (e.g., "How do I merge pathway data?"). Semantic search utilizes a `SemanticSearch` object which is shared acrosss components:

1. **Content Processing**: Load content, chunk long documents at natural boundaries
2. **Embedding Generation**: Convert chunks to 384-dim vectors using `all-MiniLM-L6-v2` sentence transformer
3. **Vector Storage**: Store in ChromaDB with metadata 
4. **Query Processing**: Embed user queries, find nearest neighbors via cosine similarity

```python
class SemanticSearch:
    def __init__(self, persist_directory: str = "./chroma_db"):
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
    
    def search(self, query: str, collection_name: str):
        # Convert query to vector, find similar content by cosine similarity
        return similarity_results_with_scores
```

## Client-Server Protocol

Here's what agents actually send and receive:

```python
# Agent request
await call_server_tool("search_tutorials", {
    "query": "consensus networks getting started", 
    "search_type": "semantic"
})

# MCP server response
{
  "results": [{
    "content": "# Merging Models into a Consensus\n\nThis tutorial shows...",
    "source": "tutorials: merging_models_into_a_consensus (part 1)",
    "similarity_score": 0.89
  }]
}
```

Agents get structured, searchable access to domain-specific knowledge - like having an expert who knows exactly where to find relevant information.

## From local to global: deployment story

### Local development

It's easy to setup a local MCP server that digests relevant documents and interacts with local agents:

```bash
# Install Napistu with MCP dependencies
pip install 'napistu[mcp]'

# Start full development server (all components)
python -m napistu.mcp server full

# Health check shows component loading
python -m napistu.mcp health --local
```

```output
🏥 Napistu MCP Server Health Check
========================================
Server URL: http://127.0.0.1:8765/mcp

Server Status: healthy

Components:
  ✅ documentation: healthy
  ✅ codebase: healthy  
  ✅ tutorials: healthy
  ✅ semantic_search: healthy
```

But this requires installing Napistu, maintaining a background process, and keeping it running - that's a lot to ask of users who just want to explore the project or collaborate on development.

### The always-up solution

Instead, I wanted an always-available service that I and others could easily use without any local setup. This meant deploying to the cloud with automatic updates whenever the codebase changes as part of my [GitHub Actions-based CI/CD workflows]([Github Actions strategy](https://github.com/napistu/napistu/wiki/GitHub-Actions-napistu%E2%80%90py))

Every new tagged version triggers deployment to Google Cloud Run:

```yaml
# Deploy workflow - simplified view
on:
  workflow_run:
    workflows: ["Release"]  # Auto-deploy after successful release
    types: [completed]
  schedule:
    - cron: '0 10 * * *'  # Daily content refresh at 2 AM PST

jobs:
  deploy:
    steps:
      - name: Deploy to Cloud Run
        run: |
          gcloud run deploy napistu-mcp-server \
            --image="us-west1-docker.pkg.dev/.../napistu-mcp-server:latest" \
            --cpu=1 --memory=2Gi \
            --set-env-vars="MCP_PROFILE=docs"
```

The production setup runs the "docs" profile (documentation + codebase + tutorials, no execution component) with 1 CPU and 2Gi memory, costing less than $1 per day. Content is refreshed nightly to capture the latest documentation changes, and health monitoring ensures automatic restarts if needed.

### The payoff

Now any AI tool can access the Napistu knowledge base instantly at https://napistu-mcp-server-844820030839.us-west1.run.app. Users don't need to install anything, run local processes, or handle maintenance - they can simply configure their AI tools to use the shared knowledge base. The service automatically updates with the latest documentation and code changes, while Cloud Run handles scaling, health checks, and automatic restarts for high availability.

```json
// Claude Desktop / Cursor configuration
{
  "mcpServers": {
    "napistu": {
      "command": "npx",
      "args": ["mcp-remote", "https://napistu-mcp-server-844820030839.us-west1.run.app/mcp/"]
    }
  }
}
```

The result: Napistu's entire knowledge base becomes instantly searchable by AI agents worldwide, dramatically lowering the barrier to contribution and collaboration.

## Lowering the Activation Energy: AI Agents in Action** *(~1200 words)*

**The Mission**: Making Napistu accessible to new users and collaborators

### **Case Study 1: Learning with Claude - "I'm new to Napistu, how do I get started with consensus networks?"**

**Without MCP**: 
- Screenshot of Claude giving generic graph theory advice
- Suggests manually browsing documentation
- No specific Napistu context or examples

**With MCP**: 
- Screenshot/video of Claude conversation showing:
  - *Callout: Claude automatically searches tutorials using MCP*
  - *Callout: Finds and references specific Napistu notebooks*
  - *Callout: Provides actual code examples from the tutorials*
- Shows Claude providing step-by-step guidance with Napistu-specific context
- Includes links to actual tutorial sections and code snippets

### **Case Study 2: Building with Cursor - "Implement a new pathway validation feature"**

**Without MCP**:
- Video/screenshots of Cursor suggesting generic validation patterns
- Misses established Napistu conventions and class structures
- Generic Python code that doesn't integrate well

**With MCP**:
- Screenshot/video of Cursor in action showing:
  - *Callout: Cursor searches codebase for existing patterns via MCP*
  - *Callout: Discovers SBML_dfs validation methods automatically*
  - *Callout: Follows established Napistu testing patterns*
- Shows generated code that properly uses Napistu classes and follows project conventions
- Demonstrates understanding of existing API patterns

**The Result**: From "intimidating research codebase" to "approachable, guided experience"

## Hard-Won Lessons: What I Learned the Hard Way

### **Gotcha #1: Semantic Search is Essential**
```python
# This didn't work well - too rigid
if "create network" in query.lower():
    return network_docs
    
# This worked much better - understands intent
semantic_results = embedding_search(query, documentation_chunks)
```
- Early exact matching was too brittle
- Users ask questions in natural language, not with precise keywords
- ChromaDB + sentence transformers made all the difference

### **Gotcha #2: AI-First Documentation**
```python
@mcp.tool()
async def search_codebase(query: str):
    """
    **USE THIS WHEN:**
    - Looking for specific Napistu functions, classes, or modules
    - Finding API documentation for Napistu features
    
    **DO NOT USE FOR:**
    - General programming concepts not specific to Napistu
    - Documentation for other libraries or frameworks
    """
```
- Initial sparse docstrings led to tool misuse
- Had to rewrite docs specifically for AI consumption
- "USE THIS WHEN" / "DO NOT USE FOR" patterns were game-changers

### **Gotcha #3: Resource vs Tool Confusion**
Show examples of agents misusing resources when they should use tools, and vice versa, until proper documentation clarified the distinction.

## The Bigger Picture: Scaling Scientific Software
- **What Worked**: Specific wins in onboarding, debugging, feature development
- **Community Building**: Lowering barriers increases contributor diversity
- **The Network Effect**: Better AI tools → more contributors → better software
- **Open Science**: Making research code truly accessible

## **VIII. Future Directions** *(~300 words)*
- **Execution Component**: Live Python environment integration (currently incomplete)
- **Multi-language Support**: Extending beyond Python to R components
- **Advanced RAG**: Better document chunking and retrieval strategies
- **Community Feedback**: Learning from early adopters

## Getting started: using and contributing to Napistu

### **For Users & Contributors**: Connect to the Napistu MCP server
```json
{
  "mcpServers": {
    "napistu": {
      "command": "npx",
      "args": ["mcp-remote", "https://napistu-mcp-server-844820030839.us-west1.run.app/mcp/"]
    }
  }
}
```

### **Try It Out**: 
- Configure Claude Desktop or Cursor with the MCP server
- Ask questions about Napistu functionality
- Start contributing to issues with AI assistance
- Join our community discussions

### **Call to Action**: 
**Help us build the future of network biology software!** The MCP server is just the beginning - we need contributors who can help extend Napistu's capabilities, improve documentation, and make systems biology research more accessible to everyone.

---

## **Key Technical Notes for Implementation:**

### **Real MCP Interactions to Demonstrate:**
1. **Tutorial Search**: Show actual semantic search finding relevant notebooks
2. **Codebase Search**: Demonstrate finding specific function signatures and docs  
3. **Documentation Retrieval**: Show how agents access wiki pages and README content
4. **Health Checks**: Demonstrate server monitoring and component status

### **Code Examples to Include:**
- Actual MCP client calls with real responses
- Before/after comparisons of AI interactions
- Configuration snippets for different AI tools
- Health check outputs showing component status

### **Story Arc:**
Problem (AI chaos) → Solution (MCP) → Implementation → Real Impact → Lessons → Community Building

**Target Audience**: Computational biologists, research software engineers, AI-assisted developers
